# Revisión de doce tópicos

Esta libreta respalda el marcador de revisión temática de la sección 4. Conserva la selección ya preparada en la comparación integrada: seis grupos por sistema, dos por banda de tamaño. No vuelve a agrupar, generar embeddings ni consultar un LLM.

**Estado inicial: evaluación humana pendiente.** Las tarjetas son evidencia para revisar; no son puntuaciones de calidad. Los comentarios humanos sobre los 13 descartes del filtro corresponden a otra evaluación.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/voc").is_dir())
NB_ROOT = REPO / "reports/difusion"
if str(NB_ROOT) not in sys.path: sys.path.insert(0, str(NB_ROOT))
from apoyo_validacion import digest, export_bundle, show_texts
from apoyo_revision import verified_bundle
plt.rcParams.update({"font.size":10,"axes.spines.top":False,"axes.spines.right":False})

In [2]:
BUNDLE = verified_bundle(NB_ROOT, "exports/comparacion-base-alternativa/20261105T191417975479Z", "4639edacc5839af99fe7121921f29e9183b320e0b5354f5090570248f3f788d1")
cases = json.loads((BUNDLE / "casos-topicos.json").read_text())
scope = json.loads((BUNDLE / "alcance.json").read_text())
selection = pd.DataFrame([{k:v for k,v in case.items() if k != "reviews"} for case in cases])

## Selección y alcance

Las bandas son terciles por rango de tamaño dentro de cada sistema. En cada banda se eligieron dos tópicos por orden SHA-256 con semilla 42. Cada tarjeta contiene hasta cinco textos distintos elegidos por hash y hasta dos evidencias guardadas adicionales. La selección está congelada; no se reemplazan los casos menos claros. Esta muestra no incluye necesariamente los grupos mayores ni representa todas las reseñas.

Las tarjetas muestran el sistema: la revisión no es ciega. Las conclusiones se limitarán a los doce grupos revisados, sin probar superioridad global.

In [3]:
display(selection[["case_id","system","band","size","keywords"]])

,case_id,system,band,size,keywords
0,base:13,base,pequeno,17,"rápida efectiva, rápida eficaz, muy eficiente,..."
1,base:16,base,pequeno,13,"útil buena, muy práctica, eficas buena, confia..."
2,base:7,base,mediano,28,"muy buena, muy bueno, buena muy, buena buena, ..."
3,base:9,base,mediano,23,"exelente exelenete, exelente exelente, mala ex..."
4,base:1,base,grande,56,"excelentes excelente, perfeccion excelente, ex..."
5,base:2,base,grande,38,"excelente aplicación, buena aplicación, aplica..."
6,alternative:26,alternative,pequeno,11,"felicidades banamex, banamex excelente, excele..."
7,alternative:19,alternative,pequeno,18,"buena, buena cool, buena buena, buena ap, ap b..."
8,alternative:13,alternative,mediano,24,"muchas fallas, tiene fallas, fallas demás, hay..."
9,alternative:16,alternative,mediano,21,"excelente atención, eficiente gracias, content..."


## Criterios de anotación

Completa `revision-manual/topicos.csv` sin alterar las columnas de identidad.

| Campo | Valores y criterio |
|---|---|
| coherencia | `2`: asunto dominante reconocible; `1`: relación parcial o varios asuntos; `0`: sin asunto compartido; `no_evaluable`: evidencia insuficiente |
| palabras_clave | `2`: describen el asunto dominante; `1`: parcialmente descriptivas o genéricas; `0`: no lo describen; `no_evaluable`: no puede decidirse |
| tipo_asunto | `funcionalidad`, `mixto`, `valoracion_general`, `otro`, `no_evaluable` |
| justificacion | Explicar el juicio con referencia a los textos, incluyendo contraejemplos |
| revisor / fecha | Autor humano y fecha de la revisión |

Una valoración general puede ser coherente sin identificar una funcionalidad. No confundimos esas dimensiones. No se rellenan automáticamente juicios humanos.

In [4]:
for case in cases:
    display(Markdown(f"### {case['case_id']} · {case['band']} · {case['size']} reseñas"))
    display(Markdown("**Palabras:** " + str(case['keywords'])))
    show_texts(case['reviews'])

### base:13 · pequeno · 17 reseñas

**Palabras:** rápida efectiva, rápida eficaz, muy eficiente, rápida axesible, mejoraron rapidez, práctica rápida, práctico eficiente, eficiente, eficiente práctica, información rápida

### base:16 · pequeno · 13 reseñas

**Palabras:** útil buena, muy práctica, eficas buena, confiable clara, práctica confiable, cómoda práctica, segura practico, siempre confiable, buena relación, usarla funcional

### base:7 · mediano · 28 reseñas

**Palabras:** muy buena, muy bueno, buena muy, buena buena, buena excelente, buena gusta, buena apiclascion, excelente tarjeta, tarjeta tengo, cambiaría nada

... (14 outputs left out)

### alternative:2 · grande · 66 reseñas

**Palabras:** app banco, app tarda, aplicación banco, banca móvil, bancaria, banca, otros bancos, entrar banca, acceso, hacer transferencias

### alternative:8 · grande · 32 reseñas

**Palabras:** mejoró app, app buena, buena app, app excelente, mejoraron aplicación, mejoró, mejorando excelente, mejor calidad, mejorado muchisimo, mejorado

## Recargar anotaciones y resumir

Guarda el CSV y vuelve a ejecutar desde esta celda. Las anotaciones existentes se conservan. Se requiere completar los doce casos antes de emitir la tabla comparativa; así evitamos presentar solo los grupos revisados primero. `no_evaluable` se cuenta aparte y no se convierte en cero.

In [5]:
from apoyo_validacion import review_sheet
review_path = NB_ROOT / "revision-manual/topicos.csv"
review = review_sheet(review_path, selection, scope['protocol_id'])
progress = review.groupby('system').agg(seleccionados=('case_id','size'), revisados=('completa','sum')).reset_index()
display(progress)
quality = pd.DataFrame()
if review.completa.all():
    rows=[]
    for system, group in review.groupby('system'):
        for criterion in ['coherencia','palabras_clave']:
            values = pd.to_numeric(group[criterion],errors='coerce')
            rows.append({'system':system,'criterio':criterion,'revisados':len(group),
                         'evaluables':int(values.notna().sum()),'no_evaluables':int(values.isna().sum()),
                         'puntuacion_0':int(values.eq(0).sum()),'puntuacion_1':int(values.eq(1).sum()),
                         'puntuacion_2':int(values.eq(2).sum())})
    quality=pd.DataFrame(rows)
    display(quality)
    display(pd.crosstab(review.system,review.tipo_asunto))
else:
    print('PENDIENTE: no se publican puntuaciones comparativas hasta revisar los doce casos.')

,system,seleccionados,revisados
0,alternative,6,0
1,base,6,0


PENDIENTE: no se publican puntuaciones comparativas hasta revisar los doce casos.


In [6]:
sources = {"frozen_cases":{"path":str((BUNDLE/'casos-topicos.json').relative_to(REPO)),"sha256":digest(BUNDLE/'casos-topicos.json')},
           "review":{"path":str(review_path.relative_to(REPO)),"sha256":digest(review_path)},
           "review_helper":{"path":"reports/difusion/apoyo_revision.py","sha256":digest(NB_ROOT/'apoyo_revision.py')}}
tables={'revision-progreso':progress,'revision-topicos':review}
if not quality.empty: tables['revision-calidad']=quality
exported=export_bundle(NB_ROOT/'revision-topicos.ipynb',{},tables,
    {'casos-topicos':cases,'estado':{'selected':len(review),'reviewed':int(review.completa.sum()),
     'state':'review_complete' if review.completa.all() else 'pending_human_review',
     'scope':'selected topics only; nonblind review; no population quality estimate'}},sources,
    'Frozen twelve-topic review; human judgments preserved, never imputed')

Evidencia exportada: /workspace/vaults/sonar/code/repos/voc-difusion/reports/difusion/exports/revision-topicos/20261106T231318700664Z
